In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

In [3]:
df = pd.read_csv(
    "../data/microsoft_file_metadata.txt",
    sep="\t",
    header=None
)

print("Shape:", df.shape)
df.head()

Shape: (3350, 13)


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,1,000e46ce402500409354,39c4a71ba8949d8e0a2d,39c4a71ba8949d8e0a2d,604879208,c,NTFS,2932253696,4310013952,126142230903260000,512,459007,502
1,7,0ae9d8ff4a2fde36885c,3455bd0b4c1501c7d899,06fcb9effa6b460f60e9,773854956,c,FAT32,5109891072,8430841856,126137892909020000,4096,6,502
2,12,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,618801528,f,NTFS,7059025920,18194284544,126134307922350000,4096,459007,198
3,13,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1878414956,c,NTFS,1083307520,2146765312,126134307738280000,512,459007,198
4,14,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1604198837,d,NTFS,5591822336,6950326272,126134307850470000,4096,459007,198


In [4]:
df.columns = [
    "SnapshotID",
    "FileID",
    "ParentDirectoryID",
    "FileNameHash",
    "FileExtension",
    "FileType",
    "Unknown",
    "FileSize",
    "AllocatedSize",
    "Timestamp",
    "ClusterSize",
    "Attributes",
    "Meta"
]

df.head()

,SnapshotID,FileID,ParentDirectoryID,FileNameHash,FileExtension,FileType,Unknown,FileSize,AllocatedSize,Timestamp,ClusterSize,Attributes,Meta
0,1,000e46ce402500409354,39c4a71ba8949d8e0a2d,39c4a71ba8949d8e0a2d,604879208,c,NTFS,2932253696,4310013952,126142230903260000,512,459007,502
1,7,0ae9d8ff4a2fde36885c,3455bd0b4c1501c7d899,06fcb9effa6b460f60e9,773854956,c,FAT32,5109891072,8430841856,126137892909020000,4096,6,502
2,12,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,618801528,f,NTFS,7059025920,18194284544,126134307922350000,4096,459007,198
3,13,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1878414956,c,NTFS,1083307520,2146765312,126134307738280000,512,459007,198
4,14,0f8bc5a91a518d0461cf,66fc28d1b3a538385a2d,eb17bfe0ed80e9c43f34,-1604198837,d,NTFS,5591822336,6950326272,126134307850470000,4096,459007,198


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3350 entries, 0 to 3349
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   SnapshotID         3350 non-null   int64 
 1   FileID             3350 non-null   object
 2   ParentDirectoryID  3350 non-null   object
 3   FileNameHash       3350 non-null   object
 4   FileExtension      3350 non-null   int64 
 5   FileType           3350 non-null   object
 6   Unknown            3350 non-null   object
 7   FileSize           3350 non-null   int64 
 8   AllocatedSize      3350 non-null   int64 
 9   Timestamp          3350 non-null   int64 
 10  ClusterSize        3350 non-null   int64 
 11  Attributes         3350 non-null   int64 
 12  Meta               3350 non-null   int64 
dtypes: int64(8), object(5)
memory usage: 340.4+ KB


In [6]:
print(df.isnull().sum())

SnapshotID           0
FileID               0
ParentDirectoryID    0
FileNameHash         0
FileExtension        0
FileType             0
Unknown              0
FileSize             0
AllocatedSize        0
Timestamp            0
ClusterSize          0
Attributes           0
Meta                 0
dtype: int64


In [7]:
numeric_cols = [
    "FileSize",
    "AllocatedSize",
    "Timestamp",
    "ClusterSize",
    "Attributes",
    "Meta"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print(df[numeric_cols].dtypes)

FileSize         int64
AllocatedSize    int64
Timestamp        int64
ClusterSize      int64
Attributes       int64
Meta             int64
dtype: object


In [8]:
df["FileDate"] = pd.to_datetime(
    (df["Timestamp"] - 116444736000000000) / 10000000,
    unit="s"
)

df["FileDate"].head()

0   2000-09-23 22:51:30.325999975
1   2000-09-18 22:21:30.901999950
2   2000-09-14 18:46:32.235000014
3   2000-09-14 18:46:13.827999949
4   2000-09-14 18:46:25.047000051
Name: FileDate, dtype: datetime64[ns]

In [9]:
df["FileSizeMB"] = (
    df["FileSize"] / (1024 * 1024)
)

df["FileSizeMB"].describe()

count      3350.000000
mean      14606.145507
std       22850.308517
min           0.007812
25%        2058.157227
50%        6767.734375
75%       18095.619141
max      424811.976562
Name: FileSizeMB, dtype: float64

In [10]:
latest_date = df["FileDate"].max()

df["FileAgeYears"] = (
    (latest_date - df["FileDate"]).dt.days
    / 365
)

df["FileAgeYears"].describe()

count    3350.000000
mean        2.361043
std         1.373310
min         0.000000
25%         1.400000
50%         2.441096
75%         3.430137
max         4.515068
Name: FileAgeYears, dtype: float64

In [11]:
df[
    [
        "FileDate",
        "FileSizeMB",
        "FileAgeYears"
    ]
].head()

,FileDate,FileSizeMB,FileAgeYears
0,2000-09-23 22:51:30.325999975,2796.415039,4.482192
1,2000-09-18 22:21:30.901999950,4873.171875,4.495890
2,2000-09-14 18:46:32.235000014,6732.011719,4.506849
3,2000-09-14 18:46:13.827999949,1033.122559,4.506849
4,2000-09-14 18:46:25.047000051,5332.777344,4.506849


In [14]:
df.to_csv(
    "../data/processed_data.csv",
    index=False
)

print("processed_data.csv saved successfully")

processed_data.csv saved successfully


In [15]:
print("\nRows:", len(df))

print("\nFile Size Statistics:")
print(df["FileSizeMB"].describe())

print("\nDate Range:")
print(df["FileDate"].min())
print(df["FileDate"].max())


Rows: 3350

File Size Statistics:
count      3350.000000
mean      14606.145507
std       22850.308517
min           0.007812
25%        2058.157227
50%        6767.734375
75%       18095.619141
max      424811.976562
Name: FileSizeMB, dtype: float64

Date Range:
2000-09-11 21:29:07.756000042
2005-03-18 00:17:51.875000


In [16]:
print(df["FileAgeYears"].quantile([0.25,0.5,0.75]))
print(df["FileSizeMB"].quantile([0.25,0.5,0.75]))

0.25    1.400000
0.50    2.441096
0.75    3.430137
Name: FileAgeYears, dtype: float64
0.25     2058.157227
0.50     6767.734375
0.75    18095.619141
Name: FileSizeMB, dtype: float64


In [17]:
age_50 = df["FileAgeYears"].quantile(0.50)
age_75 = df["FileAgeYears"].quantile(0.75)

size_75 = df["FileSizeMB"].quantile(0.75)

conditions = [
    (
        (df["FileAgeYears"] > age_75)
        &
        (df["FileSizeMB"] > size_75)
    ),

    (
        df["FileAgeYears"] > age_50
    )
]

choices = [
    "DigitalWaste",
    "Archive"
]

df["Category"] = np.select(
    conditions,
    choices,
    default="Active"
)

df["Category"] = df["Category"].astype(str)

In [18]:
print(df["Category"].value_counts())

print(
    round(
        df["Category"].value_counts(normalize=True)*100,
        2
    )
)

Category
Active          1751
Archive         1563
DigitalWaste      36
Name: count, dtype: int64
Category
Active          52.27
Archive         46.66
DigitalWaste     1.07
Name: proportion, dtype: float64


In [19]:
age_50 = df["FileAgeYears"].quantile(0.50)
age_75 = df["FileAgeYears"].quantile(0.75)

size_50 = df["FileSizeMB"].quantile(0.50)

conditions = [
    (
        (df["FileAgeYears"] > age_75)
        &
        (df["FileSizeMB"] > size_50)
    ),

    (
        df["FileAgeYears"] > age_50
    )
]

choices = [
    "DigitalWaste",
    "Archive"
]

df["Category"] = np.select(
    conditions,
    choices,
    default="Active"
)

print(df["Category"].value_counts())
print(
    round(
        df["Category"].value_counts(normalize=True)*100,
        2
    )
)

Category
Active          1751
Archive         1409
DigitalWaste     190
Name: count, dtype: int64
Category
Active          52.27
Archive         42.06
DigitalWaste     5.67
Name: proportion, dtype: float64


In [20]:
features = [
    "FileSizeMB",
    "FileAgeYears",
    "AllocatedSize",
    "ClusterSize",
    "Attributes"
]

X = df[features]
y = df["Category"]

print(X.shape)
print(y.shape)

(3350, 5)
(3350,)


In [21]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_encoded = le.fit_transform(y)

print(le.classes_)

['Active' 'Archive' 'DigitalWaste']


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(X_train.shape)
print(X_test.shape)

(2680, 5)
(670, 5)


In [23]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Training Complete")

Training Complete


In [24]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=le.classes_
    )
)

print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

      Active       1.00      1.00      1.00       350
     Archive       1.00      1.00      1.00       282
DigitalWaste       1.00      1.00      1.00        38

    accuracy                           1.00       670
   macro avg       1.00      1.00      1.00       670
weighted avg       1.00      1.00      1.00       670

[[350   0   0]
 [  0 282   0]
 [  0   0  38]]


In [25]:
import pandas as pd

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df)

         Feature  Importance
1   FileAgeYears    0.681280
0     FileSizeMB    0.197737
2  AllocatedSize    0.109900
3    ClusterSize    0.009066
4     Attributes    0.002017


In [26]:
import joblib

joblib.dump(model, "../models/digital_waste_classifier.pkl")

['../models/digital_waste_classifier.pkl']